In [2]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (adjusted_rand_score, silhouette_score,
                             confusion_matrix, accuracy_score)

In [ ]:
df = pd.read_csv('df_features.csv')

X = df.drop(columns=['label', 'source'])
y = df['label']

#transform to mean 0, variance 1
X_scaled = StandardScaler().fit_transform(X)

# PCA reduces correlated features. probably filters out rms and std after these are fitted.
X_pca = PCA(n_components=0.95, random_state=42).fit_transform(X_scaled)

#Guassian Mixture Model, covariance type tied means all components share the same covariance matrix. In documentation, there are also others. n_init, multiple restarts -> more stable result 
GMM = GaussianMixture(n_components = 2, covariance_type =  'diag', n_init = 1,  random_state = 42)
clusters = GMM.fit_predict(X_pca)

# Soft assignments (probability per cluster), a nice extra vs. K-Means
probs = GMM.predict_proba(X_pca)

In [ ]:
print("Adjusted Rand Index:", adjusted_rand_score(y, clusters)) #compares similarity of clusters
print("Silhouette score:   ", silhouette_score(X_pca, clusters)) #Looks at cluster quality. 1 is best, -1 is worst

cm = confusion_matrix(y, clusters)
print(cm)

# Match clusters to labels: flip if that gives a better score
acc = accuracy_score(y, clusters)
if acc < 0.5:
    clusters = 1 - clusters
    acc = 1 - acc
print("Accuracy after matching:", acc)

Adjusted Rand Index: 0.04637746693531368
Silhouette score:    0.48748764197551364
[[  0   0   0]
 [222  10   0]
 [159  44   0]]
Accuracy after matching: 0.9770114942528736
